# VisDrone: этап 1
Синтетические проверки и подготовка размеченных данных. GPU не нужен. Датасет скачайте самостоятельно из официального источника VisDrone и распакуйте в Google Drive. Обучение не выполняется. Holdout после подготовки не используется для выбора параметров.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Клонируйте собственный репозиторий. Для воспроизводимости зафиксируйте проверенный commit; для приватного репозитория используйте безопасную авторизацию Colab, не записывайте токен в ноутбук.

In [ ]:
!git clone https://github.com/quovadi/visdrone-4nn.git /content/visdrone-4nn
%cd /content/visdrone-4nn
!git rev-parse HEAD
%pip install -r requirements.txt
!python -m pytest -q

In [ ]:
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/VisDrone")
OUTPUT = Path("/content/drive/MyDrive/visdrone-results/data-v1")
# OUTPUT должен отсутствовать: существующий эксперимент не перезаписывается.
import json
from data.visdrone import prepare
config = json.loads(Path("configs/data.json").read_text())
report = prepare(ROOT, OUTPUT, **config)
print(json.dumps(report, indent=2))

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(OUTPUT / "train.png")))
display(Image(filename=str(OUTPUT / "validation.png")))

In [ ]:
from data.visdrone import CropRecord, VisDroneCrops, make_loader
records = [CropRecord(**r) for r in json.loads((OUTPUT / "train.json").read_text())]
dataset = VisDroneCrops(ROOT, records)
loader = make_loader(dataset, batch_size=32, workers=2, shuffle=True)
x, y = next(iter(loader))
print(x.shape, x.dtype, float(x.min()), float(x.max()), y.tolist())

Сохраните commit, manifest.json, statistics.json и версии библиотек вместе с результатами. Проверьте поддержку всех 10 классов во всех частях. Не подбирайте seed по результатам моделей. Preprocessing здесь проверочный: перед обучением будет нужен data config конкретной pretrained-модели. Хранение на Drive может замедлять чтение; при необходимости копируйте исходные split-папки в /content, сохраняя структуру.

In [ ]:
import subprocess
(OUTPUT / "environment.txt").write_text(subprocess.check_output(["python", "-m", "pip", "freeze"], text=True))
(OUTPUT / "commit.txt").write_text(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))